# Model Training and Experimentation with MLflow
Reads data from `data_db`, preprocesses it, saves the processed data back, and runs a grid search tracking at least 20 runs in MLflow.

In [1]:
import pandas as pd
import numpy as np
import mlflow
import os
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Database configuration
db_user = os.environ.get('POSTGRES_USER', 'data_user')
db_password = os.environ.get('POSTGRES_PASSWORD', 'data_password')
db_host = os.environ.get('POSTGRES_HOST', 'data_db')
db_name = os.environ.get('POSTGRES_DB', 'data_db')
db_port = '5432'
engine = create_engine(f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')

# Load raw data from database
try:
    df = pd.read_sql('SELECT * FROM raw_penguins', engine)
    print(f"Loaded {len(df)} rows from raw_penguins.")
except Exception as e:
    print(f"Could not load data. Ensure data_setup.ipynb was run. Error: {e}")

Loaded 344 rows from raw_penguins.


In [3]:
# Preprocessing
# Drop rows with missing target
df = df.dropna(subset=['species'])

# Save processed data back to database
try:
    df.to_sql('processed_penguins', engine, if_exists='replace', index=False)
    print("Saved processed data to 'processed_penguins'.")
except Exception as e:
    print(f"Error saving processed data: {e}")

X = df.drop('species', axis=1)
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Saved processed data to 'processed_penguins'.


In [4]:
# Define ML Pipeline
numeric_features = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['island', 'sex']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', RandomForestClassifier(random_state=42))])

In [5]:
# Setup MLflow
mlflow_tracking_uri = os.environ.get('MLFLOW_TRACKING_URI', 'http://mlflow:5000')
mlflow.set_tracking_uri(mlflow_tracking_uri)

experiment_name = "Penguin_Classification_GridSearch"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='s3://mlflows3/artifacts/1', creation_time=1774288551302, experiment_id='1', last_update_time=1774288551302, lifecycle_stage='active', name='Penguin_Classification_GridSearch', tags={}, workspace='default'>

In [6]:
# Define Grid Search (to ensure at least 20 runs, parameters must combine to >= 20 combinations)
# 3 n_estimators * 3 max_depth * 3 min_samples_split = 27 runs
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5, 10]
}

mlflow.autolog()

grid_search = GridSearchCV(clf, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)

print("Starting Grid Search...")
with mlflow.start_run() as run:
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    mlflow.log_metric("test_accuracy", acc)
    print(f"Best Params: {grid_search.best_params_}")
    print(f"Test Accuracy: {acc}")
    
    # Register Model
    model_name = "PenguinModel"
    # Note: MLflow autolog already logs models inside grid search, 
    # but we can explicitly log and register the best one.
    mlflow.sklearn.log_model(sk_model=best_model, artifact_path="model", registered_model_name=model_name)


2026/03/23 18:18:30 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.1.post1 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/03/23 18:18:31 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


Starting Grid Search...
Fitting 3 folds for each of 27 candidates, totalling 81 fits


2026/03/23 18:18:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/23 18:18:38 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/03/23 18:18:38 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
2026/03/23 18:18:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserializ

🏃 View run delightful-lamb-687 at: http://mlflow:5000/#/experiments/1/runs/f9b5e32f17b6427ea1d581075784e3bc
🧪 View experiment at: http://mlflow:5000/#/experiments/1
🏃 View run funny-ox-609 at: http://mlflow:5000/#/experiments/1/runs/c8899b68054243ff81d4403a5f6b4166
🧪 View experiment at: http://mlflow:5000/#/experiments/1
🏃 View run luxuriant-goose-319 at: http://mlflow:5000/#/experiments/1/runs/2cac70f3ed4d43b1befcca96a07adc17
🧪 View experiment at: http://mlflow:5000/#/experiments/1
🏃 View run resilient-snipe-317 at: http://mlflow:5000/#/experiments/1/runs/c04126a384064be5a4f102f84fbd69d4
🧪 View experiment at: http://mlflow:5000/#/experiments/1
🏃 View run adaptable-mole-550 at: http://mlflow:5000/#/experiments/1/runs/288a0efa609c446fb5b2104ca978f24c
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Best Params: {'classifier__max_depth': None, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Test Accuracy: 0.9855072463768116


Registered model 'PenguinModel' already exists. Creating a new version of this model...
2026/03/23 18:18:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PenguinModel, version 2


🏃 View run lyrical-roo-684 at: http://mlflow:5000/#/experiments/1/runs/6d0af2db2fa4427089d8ce0643cd5a85
🧪 View experiment at: http://mlflow:5000/#/experiments/1


Created version '2' of model 'PenguinModel'.
